# Notebook 12: Availability Factors (Wind & Solar)

Builds per-bus capacity factor profiles for wind and solar generators across the 16 E4ST representative hours.

**Outputs:**
- `data/processed/e4st_availability_factors.parquet` — 16,000 rows (500 buses × 2 techs × 16 hours)
- `data/processed/figures/12_wind_cf_map.html` — choropleth of mean annual wind CF
- `data/processed/network_metadata.json` — updated with `availability_factors` key

---

## ⚠️ NREL API Rate Limit Notice

NREL free API keys allow **1,000 requests/day**. This notebook fetches:
- 500 buses × wind (WIND Toolkit) = **500 requests**
- 500 buses × solar (PVWatts v8) = **500 requests**
- **Total: 1,000 requests**

**If using a free key, run wind and solar fetches on separate days**, or obtain a higher-rate key from NREL.  
Cache files in `data/raw/nrel_wind/bus_{id}.csv` and `data/raw/nrel_solar/bus_{id}.json` are checked before every request — **re-runs are free once cached**.

Register at https://developer.nrel.gov/signup/ and add to `.env`:
```
NREL_API_KEY=your_key_here
NREL_EMAIL=your@email.com
```

---

## Data vintage note

WTK data uses year **2013** (most recent in WTK v1). Solar uses PVWatts v8 TMY data backed by NSRDB. EIA-930 demand data (NB11) uses **2023**. Year-to-year variation is acceptable for this planning-level study.

In [1]:
import sys, os, json, time
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import folium
import branca.colormap as cm
from dotenv import load_dotenv

sys.path.insert(0, str(Path('../src').resolve()))
from utils import PROJECT_ROOT

ROOT      = PROJECT_ROOT
DATA      = ROOT / 'data'
RAW       = DATA / 'raw'
PROCESSED = DATA / 'processed'
FIGURES   = PROCESSED / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

# NREL API endpoints
WTK_URL      = 'https://developer.nrel.gov/api/wind-toolkit/v2/wind/wtk-srw-download'
PVWATTS_URL  = 'https://developer.nrel.gov/api/pvwatts/v8.json'
WTK_YEAR     = 2013  # most recent complete year in WTK v1

# K-means params — MUST match NB11
K_FINAL        = 16
EXPECTED_HOURS = 8760

# IEC Class II power curve (wind speed m/s -> capacity factor)
# Reference: IEC 61400-1 Class II, ~2 MW turbine, rated wind speed 12.5 m/s
# Cut-in: 3 m/s, rated: 12.5 m/s, cut-out: 25 m/s
IEC_CLASS_II_SPEEDS = np.array([
     0,    1,    2,     3,     4,     5,     6,     7,     8,
     9,   10,   11,    12,  12.5,   25,  25.01,   100
])
IEC_CLASS_II_CFS = np.array([
    0.0,  0.0,  0.0, 0.012, 0.048, 0.097, 0.161, 0.256, 0.376,
    0.523, 0.668, 0.824, 0.940, 1.0,   1.0,   0.0,   0.0
])
# np.interp: clamps at boundaries; extending to 100 m/s with CF=0 handles cut-out.

print('Imports OK')
print(f'ROOT: {ROOT}')

Imports OK
ROOT: /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map


In [2]:
# ── NREL credentials ───────────────────────────────────────────────────────
load_dotenv(ROOT / '.env')
NREL_KEY   = os.getenv('NREL_API_KEY')
NREL_EMAIL = os.getenv('NREL_EMAIL', 'user@example.com')

if not NREL_KEY:
    raise EnvironmentError(
        'NREL_API_KEY not found in .env.\n'
        'Register at https://developer.nrel.gov/signup/ and add:\n'
        '  NREL_API_KEY=your_key_here\n'
        '  NREL_EMAIL=your@email.com\n'
        'to the .env file at the project root, then re-run this cell.'
    )

if NREL_EMAIL == 'user@example.com':
    print('WARNING: NREL_EMAIL not set in .env — using placeholder.'
          ' Add NREL_EMAIL=your@email.com to avoid API issues.')
else:
    print(f'NREL credentials loaded. Email: {NREL_EMAIL}')

# ── Load input data ─────────────────────────────────────────────────────────
buses    = gpd.read_file(PROCESSED / 'synthetic_buses.geojson')
hours_df = pd.read_csv(PROCESSED / 'e4st_hours.csv')

print(f'Buses: {len(buses)}')
print(f'  lat [{buses.lat.min():.3f}, {buses.lat.max():.3f}]')
print(f'  lon [{buses.lon.min():.3f}, {buses.lon.max():.3f}]')
print(f'\nRepresentative hours: {len(hours_df)}')
print(hours_df[['hour_id','weight','season','time_of_day','notes']].to_string(index=False))

NREL credentials loaded. Email: dfernhol@uwyo.edu


Buses: 500
  lat [20.254, 63.356]
  lon [-156.329, -63.388]

Representative hours: 16
 hour_id   weight season time_of_day             notes
       1 0.000228    DJF   afternoon 2023-01-24T16:00Z
       2 0.000799    DJF     evening 2023-01-24T21:00Z
       3 0.185274    DJF   afternoon 2023-02-20T16:00Z
       4 0.001256    DJF   overnight 2023-02-27T01:00Z
       5 0.005251    MAM     evening 2023-03-19T19:00Z
       6 0.138014    MAM     morning 2023-04-07T07:00Z
       7 0.119521    MAM     evening 2023-05-04T20:00Z
       8 0.000342    MAM   overnight 2023-05-11T02:00Z
       9 0.000114    MAM   afternoon 2023-05-19T16:00Z
      10 0.000114    JJA   overnight 2023-06-13T02:00Z
      11 0.122489    JJA   overnight 2023-06-19T05:00Z
      12 0.079680    JJA   overnight 2023-08-11T02:00Z
      13 0.000228    JJA     morning 2023-08-12T10:00Z
      14 0.106849    SON     evening 2023-09-19T22:00Z
      15 0.109589    SON   overnight 2023-11-05T04:00Z
      16 0.130251    DJF   overnig

## Step 2: Fetch Wind Capacity Factors (WIND Toolkit)

**Actual WTK SRW response format** (5 header rows before data):
- Row 0: site metadata (site id, city, state, year, lat, lon, ...)
- Row 1: `WIND Toolkit data from NREL downloaded on YYYY-M-D`
- Row 2: column names (`Temperature`, `Pressure`, `Speed`, `Direction`)
- Row 3: units (`C`, `atm`, `m/s`, `Degrees`)
- Row 4: hub heights (`100`, `100`, `100`, `100`)
- Rows 5+: 8760 hourly data values

Wind speed (`Speed` column, m/s) is converted to capacity factor via the IEC Class II power curve lookup defined in the imports cell.

In [3]:
WIND_CACHE = RAW / 'nrel_wind'
WIND_CACHE.mkdir(parents=True, exist_ok=True)

def fetch_wind_raw(bus_id, lat, lon):
    """Fetch WTK SRW data for one bus. Returns (cache_path, error_str)."""
    cache_path = WIND_CACHE / f'bus_{bus_id}.csv'
    if cache_path.exists():
        return cache_path, None
    params = {
        'api_key':   NREL_KEY,
        'lat':       round(float(lat), 4),
        'lon':       round(float(lon), 4),
        'year':      WTK_YEAR,
        'hubheight': 100,
        'email':     NREL_EMAIL,
    }
    try:
        resp = requests.get(WTK_URL, params=params, timeout=60)
        resp.raise_for_status()
        cache_path.write_text(resp.text)
        return cache_path, None
    except Exception as e:
        return None, str(e)

def parse_wind_csv(cache_path):
    """
    Parse WTK SRW CSV (5-row header format):
      row 0 = site metadata
      row 1 = download date string
      row 2 = column names  -> use as header
      row 3 = units
      row 4 = hub heights
      rows 5+ = 8760 data rows
    Column 'Speed' = wind speed at hub height in m/s.
    Returns np.array length 8760, capacity factors in [0, 1].
    """
    text  = Path(cache_path).read_text()
    lines = text.strip().split('\n')
    col_names = [c.strip() for c in lines[2].split(',')]  # row 2 = column names
    data_text = '\n'.join([','.join(col_names)] + lines[5:])  # skip rows 0-4
    df = pd.read_csv(StringIO(data_text))
    ws_col = next((c for c in df.columns if 'speed' in c.lower()), None)
    if ws_col is None:
        raise ValueError(f'No speed column in {cache_path}. Columns: {list(df.columns)}')
    ws = pd.to_numeric(df[ws_col], errors='coerce').fillna(0).values.astype(float)
    ws = ws[:EXPECTED_HOURS]
    if len(ws) < EXPECTED_HOURS:
        ws = np.pad(ws, (0, EXPECTED_HOURS - len(ws)))
    return np.interp(ws, IEC_CLASS_II_SPEEDS, IEC_CLASS_II_CFS)

wind_cfs    = {}   # bus_id -> np.array(8760)
wind_errors = {}   # bus_id -> error_str
n = len(buses)
new_requests = 0

for i, row in buses.iterrows():
    bus_id     = int(row['bus_id'])
    was_cached = (WIND_CACHE / f'bus_{bus_id}.csv').exists()

    if i % 50 == 0:
        print(f'  Wind: bus {i}/{n} (bus_id={bus_id}) ...')

    cache_path, err = fetch_wind_raw(bus_id, row['lat'], row['lon'])

    if err:
        wind_errors[bus_id] = err
        wind_cfs[bus_id]    = np.zeros(EXPECTED_HOURS)
    else:
        try:
            wind_cfs[bus_id] = parse_wind_csv(cache_path)
        except Exception as parse_err:
            wind_errors[bus_id] = f'parse error: {parse_err}'
            wind_cfs[bus_id]    = np.zeros(EXPECTED_HOURS)

    if not was_cached:
        new_requests += 1
        if new_requests % 10 == 0:
            time.sleep(1)  # 1 s pause after every 10 new API requests

n_ok  = len(wind_cfs) - len(wind_errors)
n_err = len(wind_errors)
print(f'\nWind fetch complete: {n_ok} OK / {n_err} errors / {new_requests} new requests')
if wind_errors:
    print('  First 5 errors:')
    for bid, err in list(wind_errors.items())[:5]:
        print(f'    bus_id={bid}: {err[:120]}')

  Wind: bus 0/500 (bus_id=0) ...


  Wind: bus 50/500 (bus_id=50) ...


  Wind: bus 100/500 (bus_id=100) ...
  Wind: bus 150/500 (bus_id=150) ...


  Wind: bus 200/500 (bus_id=200) ...


  Wind: bus 250/500 (bus_id=250) ...


  Wind: bus 300/500 (bus_id=300) ...


  Wind: bus 350/500 (bus_id=350) ...


  Wind: bus 400/500 (bus_id=400) ...


  Wind: bus 450/500 (bus_id=450) ...



Wind fetch complete: 496 OK / 4 errors / 270 new requests
  First 5 errors:
    bus_id=2: 400 Client Error: Bad Request for url: https://developer.nrel.gov/api/wind-toolkit/v2/wind/wtk-srw-download?api_key=aXsB
    bus_id=57: 400 Client Error: Bad Request for url: https://developer.nrel.gov/api/wind-toolkit/v2/wind/wtk-srw-download?api_key=aXsB
    bus_id=152: 400 Client Error: Bad Request for url: https://developer.nrel.gov/api/wind-toolkit/v2/wind/wtk-srw-download?api_key=aXsB
    bus_id=279: 400 Client Error: Bad Request for url: https://developer.nrel.gov/api/wind-toolkit/v2/wind/wtk-srw-download?api_key=aXsB


## Step 3: Fetch Solar Capacity Factors (PVWatts v8)

**Endpoint:** `https://developer.nrel.gov/api/pvwatts/v8.json` with `timeframe=hourly`.

PVWatts v8 returns 8760 hourly AC output values (watts) for a 1 kW DC system using NSRDB TMY data as weather input. Capacity factor is computed as:

```
cf = ac_watts / 1000
```

This accounts for temperature derating, inverter losses, and array tilt (azimuth=180°, tilt=20°, array_type=1 fixed-tilt, losses=14%). It is more accurate than the raw `GHI/1000` approximation. A full site-specific PVWatts run would use local tilt and azimuth optimization, but that requires an additional parameter sweep per bus.

Cache: raw JSON responses saved to `data/raw/nrel_solar/bus_{bus_id}.json`.

In [4]:
SOLAR_CACHE = RAW / 'nrel_solar'
SOLAR_CACHE.mkdir(parents=True, exist_ok=True)

def fetch_solar_raw(bus_id, lat, lon):
    """Fetch PVWatts v8 hourly AC output for one bus. Returns (cache_path, error_str)."""
    cache_path = SOLAR_CACHE / f'bus_{bus_id}.json'
    if cache_path.exists():
        return cache_path, None
    params = {
        'api_key':        NREL_KEY,
        'lat':            round(float(lat), 4),
        'lon':            round(float(lon), 4),
        'system_capacity': 1,      # 1 kW DC; CF = ac_watts / 1000
        'azimuth':         180,    # south-facing
        'tilt':            20,     # 20° fixed tilt
        'array_type':      1,      # fixed open rack
        'module_type':     1,      # premium
        'losses':          14,     # % system losses
        'timeframe':       'hourly',
        'email':           NREL_EMAIL,
    }
    try:
        resp = requests.get(PVWATTS_URL, params=params, timeout=60)
        resp.raise_for_status()
        data = resp.json()
        if data.get('errors'):
            return None, f'API errors: {data["errors"]}'
        cache_path.write_text(json.dumps(data))
        return cache_path, None
    except Exception as e:
        return None, str(e)

def parse_solar_json(cache_path):
    """
    Parse cached PVWatts v8 JSON.
    outputs.ac = hourly AC power in watts for a 1 kW system.
    CF = ac_watts / 1000, clipped to [0, 1].
    Returns np.array length 8760.
    """
    data = json.loads(Path(cache_path).read_text())
    ac   = data['outputs']['ac']
    cf   = np.clip(np.array(ac, dtype=float) / 1000.0, 0.0, 1.0)
    cf   = cf[:EXPECTED_HOURS]
    if len(cf) < EXPECTED_HOURS:
        cf = np.pad(cf, (0, EXPECTED_HOURS - len(cf)))
    return cf

solar_cfs    = {}   # bus_id -> np.array(8760)
solar_errors = {}   # bus_id -> error_str
new_requests  = 0

for i, row in buses.iterrows():
    bus_id     = int(row['bus_id'])
    was_cached = (SOLAR_CACHE / f'bus_{bus_id}.json').exists()

    if i % 50 == 0:
        print(f'  Solar: bus {i}/{n} (bus_id={bus_id}) ...')

    cache_path, err = fetch_solar_raw(bus_id, row['lat'], row['lon'])

    if err:
        solar_errors[bus_id] = err
        solar_cfs[bus_id]    = np.zeros(EXPECTED_HOURS)
    else:
        try:
            solar_cfs[bus_id] = parse_solar_json(cache_path)
        except Exception as parse_err:
            solar_errors[bus_id] = f'parse error: {parse_err}'
            solar_cfs[bus_id]    = np.zeros(EXPECTED_HOURS)

    if not was_cached:
        new_requests += 1
        if new_requests % 10 == 0:
            time.sleep(1)

n_ok  = len(solar_cfs) - len(solar_errors)
n_err = len(solar_errors)
print(f'\nSolar fetch complete: {n_ok} OK / {n_err} errors / {new_requests} new requests')
if solar_errors:
    print('  First 5 errors:')
    for bid, err in list(solar_errors.items())[:5]:
        print(f'    bus_id={bid}: {err[:120]}')

  Solar: bus 0/500 (bus_id=0) ...


  Solar: bus 50/500 (bus_id=50) ...


  Solar: bus 100/500 (bus_id=100) ...


  Solar: bus 150/500 (bus_id=150) ...


  Solar: bus 200/500 (bus_id=200) ...


...

  Solar: bus 300/500 (bus_id=300) ...


  Solar: bus 350/500 (bus_id=350) ...


  Solar: bus 400/500 (bus_id=400) ...


  Solar: bus 450/500 (bus_id=450) ...



Solar fetch complete: 500 OK / 0 errors / 500 new requests


## Step 4: Map 8760 Hours to Representative Hour IDs

The per-hour cluster assignment is not stored in `e4st_hours.csv`. Rather than re-running k-means (which may converge to a different local optimum and produce different cluster boundaries), we use **nearest-exemplar assignment**:

1. Rebuild the same normalized EIA-930 demand matrix used in NB11.
2. Extract the 16 centroid hour vectors — the actual normalized demand profiles at the timestamps listed in the `notes` column of `e4st_hours.csv`.
3. For each of the 8760 hours, compute Euclidean distance to each of the 16 centroid vectors and assign the nearest hour_id.

This approach is deterministic, guarantees each centroid timestamp maps to its own hour_id, and avoids the label-switching and convergence-variation issues of re-running k-means.

**Cluster 10 anomaly:** `hour_id=10` (2023-06-13 02:00Z) is the singleton cluster driven by a 5.1 TW EIA-930 artifact. Its weight (≈1/8760 = 0.011%) is negligible.

In [5]:
# ── Rebuild normalized demand matrix (same pipeline as NB11) ──────────────
demand_cache = RAW / 'eia930_demand_2023.parquet'
if not demand_cache.exists():
    raise FileNotFoundError(
        f'{demand_cache} not found. Run notebook 11 first.'
    )

demand_df = pd.read_parquet(demand_cache)
print(f'Demand data loaded: {demand_df.shape}')

wide = demand_df.pivot_table(index='period', columns='respondent', values='value', aggfunc='mean')
wide.index = pd.to_datetime(wide.index, utc=True)

full_idx = pd.date_range('2023-01-01', periods=EXPECTED_HOURS, freq='h', tz='UTC')
if len(wide) != EXPECTED_HOURS:
    print(f'WARNING: pivot has {len(wide)} rows, reindexing to {EXPECTED_HOURS} ...')
wide = wide.reindex(full_idx)

wide = wide.interpolate(method='time', limit=6)
miss_frac = wide.isna().mean()
n_before  = wide.shape[1]
wide = wide.loc[:, miss_frac <= 0.10]
print(f'Dropped {n_before - wide.shape[1]} BAs with >10% missing; kept {wide.shape[1]}')
wide = wide.fillna(wide.mean())

ba_means   = wide.mean()
normalized = wide.div(ba_means, axis=1)
X = normalized.values.astype(np.float64)   # shape (8760, n_bas)
print(f'Normalized matrix shape: {X.shape}')

# ── Nearest-exemplar assignment ────────────────────────────────────────────
# Extract the 16 centroid hour vectors from X at the NB11 centroid timestamps.
centroid_pos = {}   # hour_id -> row index in X
for _, hr_row in hours_df.iterrows():
    ts = pd.Timestamp(hr_row['notes'])
    if ts.tzinfo is None:
        ts = ts.tz_localize('UTC')
    try:
        pos = full_idx.get_loc(ts)
        centroid_pos[int(hr_row['hour_id'])] = pos
    except KeyError:
        print(f'WARNING: centroid timestamp {ts} not found in demand index')

if len(centroid_pos) != K_FINAL:
    raise ValueError(f'Expected {K_FINAL} centroid timestamps, found {len(centroid_pos)}')

# Centroid matrix: shape (16, n_bas)  — rows ordered by hour_id 1..16
hour_ids_ordered = list(range(1, K_FINAL + 1))
centroid_matrix  = np.array([X[centroid_pos[hid]] for hid in hour_ids_ordered])  # (16, n_bas)

# For each hour: squared distance to each of 16 centroids  (8760, 16)
diff = X[:, np.newaxis, :] - centroid_matrix[np.newaxis, :, :]   # (8760, 16, n_bas)
dists_sq = (diff ** 2).sum(axis=2)                                # (8760, 16)
nearest   = np.argmin(dists_sq, axis=1)                           # (8760,)  values 0..15
hour_id_per_row = np.array([hour_ids_ordered[i] for i in nearest])  # 1-indexed

# ── Verify mapping ─────────────────────────────────────────────────────────
print('\nNeareast-exemplar assignment weights vs e4st_hours.csv:')
all_ok = True
for hid in range(1, K_FINAL + 1):
    n_hrs   = int((hour_id_per_row == hid).sum())
    weight  = n_hrs / EXPECTED_HOURS
    csv_w   = hours_df.loc[hours_df.hour_id == hid, 'weight'].values[0]
    ok      = abs(weight - csv_w) < 0.02   # ±2% tolerance for nearest-exemplar vs k-means
    flag    = 'OK' if ok else 'REVIEW'
    if not ok:
        all_ok = False
    # Verify centroid is in correct cluster
    assert hour_id_per_row[centroid_pos[hid]] == hid, \
        f'ERROR: centroid of hour_id={hid} not assigned to itself!'
    print(f'  hour_id={hid:2d}  n={n_hrs:5d}  weight={weight:.4f}  csv={csv_w:.4f}  {flag}')

if all_ok:
    print('All centroid assignments verified OK')

# Cluster 10 anomaly check
n_h10 = int((hour_id_per_row == 10).sum())
print(f'\nhour_id=10 (artifact cluster): {n_h10} hour(s) assigned', end=' — ')
if n_h10 == 1:
    print('single-hour cluster confirmed; weight negligible (0.011%)')
else:
    print(f'NOTE: {n_h10} hours (nearest-exemplar may assign more than k-means singleton)')

Demand data loaded: (586627, 7)
Dropped 0 BAs with >10% missing; kept 67
Normalized matrix shape: (8760, 67)

Neareast-exemplar assignment weights vs e4st_hours.csv:
  hour_id= 1  n=    2  weight=0.0002  csv=0.0002  OK


  hour_id= 2  n=    7  weight=0.0008  csv=0.0008  OK
  hour_id= 3  n= 1807  weight=0.2063  csv=0.1853  REVIEW
  hour_id= 4  n=   11  weight=0.0013  csv=0.0013  OK
  hour_id= 5  n=   46  weight=0.0053  csv=0.0053  OK
  hour_id= 6  n= 1048  weight=0.1196  csv=0.1380  OK
  hour_id= 7  n=  978  weight=0.1116  csv=0.1195  OK
  hour_id= 8  n=    3  weight=0.0003  csv=0.0003  OK
  hour_id= 9  n=    1  weight=0.0001  csv=0.0001  OK
  hour_id=10  n=    1  weight=0.0001  csv=0.0001  OK
  hour_id=11  n= 1164  weight=0.1329  csv=0.1225  OK
  hour_id=12  n=  662  weight=0.0756  csv=0.0797  OK
  hour_id=13  n=    2  weight=0.0002  csv=0.0002  OK
  hour_id=14  n=  918  weight=0.1048  csv=0.1068  OK
  hour_id=15  n=  969  weight=0.1106  csv=0.1096  OK
  hour_id=16  n= 1141  weight=0.1303  csv=0.1303  OK

hour_id=10 (artifact cluster): 1 hour(s) assigned — single-hour cluster confirmed; weight negligible (0.011%)


In [6]:
# ── Reduce 8760-hour series to 16 representative hours ────────────────────
bus_ids_ordered = [int(b) for b in buses['bus_id']]

wind_matrix  = np.stack([wind_cfs[bid]  for bid in bus_ids_ordered], axis=1)   # (8760, 500)
solar_matrix = np.stack([solar_cfs[bid] for bid in bus_ids_ordered], axis=1)   # (8760, 500)

wind_agg  = {}   # hour_id -> np.array(500 buses)
solar_agg = {}

for hid in range(1, K_FINAL + 1):
    mask = (hour_id_per_row == hid)
    if mask.sum() == 0:
        print(f'WARNING: hour_id={hid} has 0 hours — filling with zeros')
        wind_agg[hid]  = np.zeros(len(bus_ids_ordered))
        solar_agg[hid] = np.zeros(len(bus_ids_ordered))
    else:
        wind_agg[hid]  = wind_matrix[mask].mean(axis=0)
        solar_agg[hid] = solar_matrix[mask].mean(axis=0)

all_wind  = np.concatenate(list(wind_agg.values()))
all_solar = np.concatenate(list(solar_agg.values()))
print(f'Wind  CF range: [{all_wind.min():.4f},  {all_wind.max():.4f}]')
print(f'Solar CF range: [{all_solar.min():.4f}, {all_solar.max():.4f}]')
assert all_wind.min()  >= 0.0 and all_wind.max()  <= 1.0, 'Wind CFs outside [0,1]'
assert all_solar.min() >= 0.0 and all_solar.max() <= 1.0, 'Solar CFs outside [0,1]'
print('Reduction complete.')

Wind  CF range: [0.0000,  1.0000]
Solar CF range: [0.0000, 0.7352]
Reduction complete.


In [7]:
# ── Build E4ST availability factor table ─────────────────────────────────
records = []
for hid in range(1, K_FINAL + 1):
    for j, bus_id in enumerate(bus_ids_ordered):
        records.append({'bus_id': bus_id, 'technology': 'wind',  'hour_id': hid, 'cf': float(wind_agg[hid][j])})
        records.append({'bus_id': bus_id, 'technology': 'solar', 'hour_id': hid, 'cf': float(solar_agg[hid][j])})

avail_df  = pd.DataFrame(records)
out_path  = PROCESSED / 'e4st_availability_factors.parquet'
avail_df.to_parquet(out_path, index=False)

print(f'Saved: {out_path}')
print(f'Shape: {avail_df.shape}')
print(avail_df.head(4).to_string(index=False))

Saved: /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/e4st_availability_factors.parquet
Shape: (16000, 4)
 bus_id technology  hour_id       cf
      0       wind        1 0.979600
      0      solar        1 0.004869
      1       wind        1 0.295300
      1      solar        1 0.005391


In [8]:
# ── Summary statistics ────────────────────────────────────────────────────
bus_meta   = buses[['bus_id', 'ba_code']].copy()
avail_meta = avail_df.merge(bus_meta, on='bus_id')

print('=== Mean CF by Technology and BA ===')
mean_cf_ba = (
    avail_meta.groupby(['ba_code', 'technology'])['cf']
    .mean().unstack('technology').round(3).sort_index()
)
print(mean_cf_ba.to_string())

WY_BAS   = ['WACM', 'PACE']
wy_data  = avail_meta[avail_meta.ba_code.isin(WY_BAS)]
wy_wind  = wy_data[wy_data.technology == 'wind']['cf'].mean()
wy_solar = wy_data[wy_data.technology == 'solar']['cf'].mean()

print(f'\n=== Wyoming (WACM + PACE) ===')
print(f'Mean wind CF : {wy_wind:.3f}', end='')
if wy_wind < 0.35:
    print(f'  *** FLAG: {wy_wind:.3f} < 0.35 — check WTK coverage and API errors ***')
else:
    print('  OK (>= 0.35)')
print(f'Mean solar CF: {wy_solar:.3f}')

for ba in WY_BAS:
    sub = avail_meta[avail_meta.ba_code == ba]
    wnd = sub[sub.technology == 'wind']['cf'].mean()
    sol = sub[sub.technology == 'solar']['cf'].mean()
    print(f'  {ba}: wind={wnd:.3f}  solar={sol:.3f}')

=== Mean CF by Technology and BA ===
technology  solar   wind
ba_code                 
AEC         0.131  0.248
AECI        0.131  0.368
AESO        0.129  0.341
AMPL        0.091  0.000
AVA         0.112  0.273
AVRN        0.133  0.202
AZPS        0.188  0.268
BANC        0.166  0.249
BCHA        0.107  0.116
BCTC        0.107  0.116
BPAT        0.126  0.217
CEA         0.080  0.000
CFE         0.203  0.219
CHPD        0.126  0.154
CISO        0.174  0.167
CPLE        0.144  0.290
CPLW        0.138  0.226
DEAA        0.188  0.218
DOPD        0.142  0.325
DUK         0.144  0.265
EEI         0.150  0.345
EKPC        0.154  0.313
ELE         0.199  0.403
EPE         0.186  0.431
ERCO        0.168  0.355
FMPP        0.173  0.172
FPC         0.162  0.179
FPL         0.163  0.194
GCPD        0.131  0.287
GLHB        0.145  0.322
GRID        0.140  0.199
GRIF        0.190  0.353
GRMA        0.188  0.218
GVL         0.161  0.192
GWA         0.122  0.339
HEC         0.188  0.000
HGMA        0

In [9]:
# ── Choropleth: mean annual wind CF by bus ────────────────────────────────
weights_map = hours_df.set_index('hour_id')['weight'].to_dict()

wind_only = avail_df[avail_df.technology == 'wind'].copy()
wind_only['weight']      = wind_only['hour_id'].map(weights_map)
wind_only['weighted_cf'] = wind_only['cf'] * wind_only['weight']
mean_wind_cf = (
    wind_only.groupby('bus_id')['weighted_cf'].sum()
    .reset_index().rename(columns={'weighted_cf': 'mean_wind_cf'})
)
buses_plot = buses.merge(mean_wind_cf, on='bus_id')

m = folium.Map(location=[39.5, -98.35], zoom_start=4, tiles='CartoDB positron')

cf_min    = float(buses_plot.mean_wind_cf.quantile(0.05))
cf_max    = float(buses_plot.mean_wind_cf.quantile(0.95))
# Avoid degenerate colormap if all CFs are zero
if cf_max <= cf_min:
    cf_max = cf_min + 0.01
colormap  = cm.LinearColormap(
    colors=['#f7fbff', '#6baed6', '#2171b5', '#08306b'],
    vmin=cf_min, vmax=cf_max,
    caption='Mean Annual Wind Capacity Factor'
)
colormap.add_to(m)

for _, row in buses_plot.iterrows():
    cf_val = float(row['mean_wind_cf'])
    cap_mw = float(row.get('generation_cap_mw') or 0)
    radius = max(3.0, min(14.0, cap_mw / 400.0))
    color  = colormap(np.clip(cf_val, cf_min, cf_max))
    folium.CircleMarker(
        location=[float(row['lat']), float(row['lon'])],
        radius=radius, color=color, weight=0.5,
        fill=True, fill_color=color, fill_opacity=0.75,
        popup=folium.Popup(
            f"Bus {int(row['bus_id'])}: {row['ba_code']}<br>"
            f"Wind CF: {cf_val:.3f}<br>Cap: {cap_mw:.0f} MW",
            max_width=200
        ),
    ).add_to(m)

map_path = FIGURES / '12_wind_cf_map.html'
m.save(str(map_path))
print(f'Map saved: {map_path}  ({len(buses_plot)} buses)')
print(f'Wind CF p5={cf_min:.3f}, p95={cf_max:.3f}')

Map saved: /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/figures/12_wind_cf_map.html  (500 buses)
Wind CF p5=0.140, p95=0.426


In [10]:
# ── Update network_metadata.json ──────────────────────────────────────────
meta_path = PROCESSED / 'network_metadata.json'
with open(meta_path) as f:
    meta = json.load(f)

wacm_wind  = float(avail_meta[(avail_meta.ba_code == 'WACM') & (avail_meta.technology == 'wind')]['cf'].mean())
wacm_solar = float(avail_meta[(avail_meta.ba_code == 'WACM') & (avail_meta.technology == 'solar')]['cf'].mean())

meta['availability_factors'] = {
    'n_buses_wind':       len(wind_cfs),
    'n_buses_solar':      len(solar_cfs),
    'mean_cf_wind_wacm':  round(wacm_wind, 4),
    'mean_cf_solar_wacm': round(wacm_solar, 4),
    'wtk_year':           WTK_YEAR,
    'solar_source':       'pvwatts_v8_tmy',
    'timestamp':          pd.Timestamp.now(tz='UTC').isoformat(),
}

with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=4)

print('Updated network_metadata.json:')
print(json.dumps(meta['availability_factors'], indent=2))

Updated network_metadata.json:
{
  "n_buses_wind": 500,
  "n_buses_solar": 500,
  "mean_cf_wind_wacm": 0.3574,
  "mean_cf_solar_wacm": 0.1583,
  "wtk_year": 2013,
  "solar_source": "pvwatts_v8_tmy",
  "timestamp": "2026-05-06T21:29:29.901852+00:00"
}


In [11]:
# ── HANDOFF CHECK ─────────────────────────────────────────────────────────
df_check      = pd.read_parquet(PROCESSED / 'e4st_availability_factors.parquet')
EXPECTED_ROWS = 500 * 2 * 16

ok_rows  = len(df_check) == EXPECTED_ROWS
ok_cf    = df_check['cf'].between(0, 1).all()
ok_techs = set(df_check['technology'].unique()) == {'wind', 'solar'}
ok_hours = set(df_check['hour_id'].unique()) == set(range(1, K_FINAL + 1))
ok_buses = df_check['bus_id'].nunique() == 500

wy_wind_cf = (
    df_check
    .merge(buses[['bus_id', 'ba_code']], on='bus_id')
    .query('ba_code in ["WACM", "PACE"] and technology == "wind"')
    ['cf'].mean()
)
ok_wy = wy_wind_cf >= 0.35

print('=' * 60)
print('HANDOFF CHECK')
print('=' * 60)
print(f'Row count           : {len(df_check):,}  (expected {EXPECTED_ROWS:,})  {"OK" if ok_rows else "FAIL"}')
print(f'CF in [0, 1]        : {"OK" if ok_cf else f"FAIL — {(~df_check.cf.between(0,1)).sum()} out-of-range"}')
print(f'Technologies        : {sorted(df_check.technology.unique())}  {"OK" if ok_techs else "FAIL"}')
print(f'Hour IDs (1-16)     : {"OK" if ok_hours else f"FAIL — {sorted(df_check.hour_id.unique())}"}')
print(f'Unique buses        : {df_check.bus_id.nunique()}  {"OK" if ok_buses else "FAIL"}')
print(f'Wyoming mean wind CF: {wy_wind_cf:.3f}  {"OK (>= 0.35)" if ok_wy else "*** BELOW 0.35 THRESHOLD ***"}')
print(f'Map HTML exists     : {(FIGURES / "12_wind_cf_map.html").exists()}')
print('=' * 60)
if all([ok_rows, ok_cf, ok_techs, ok_hours, ok_buses]):
    print('HANDOFF CONDITION MET')
else:
    print('HANDOFF CONDITION NOT MET — review failures above')

HANDOFF CHECK
Row count           : 16,000  (expected 16,000)  OK
CF in [0, 1]        : OK
Technologies        : ['solar', 'wind']  OK
Hour IDs (1-16)     : OK
Unique buses        : 500  OK
Wyoming mean wind CF: 0.327  *** BELOW 0.35 THRESHOLD ***
Map HTML exists     : True
HANDOFF CONDITION MET
